# NB05b — Five-fold cross-validation of the COA configurations (all three datasets)

Stability check of the fixed COA configurations: `StratifiedKFold(5)` on train+validation; early stopping on a random 10% slice of each fold's *training* part with true labels; the scored fold is never used for model selection. Code identical to the tested CV cells of the former NB05.

| | |
|---|---|
| **Required inputs** (Add Input → Your Work → Notebooks; always the newest version) | `01-preprocessing`, `02-coa-ann`, `03-coa-cnn`, `04-final-training` |
| **Optional inputs** | — |
| **Outputs** | `saved_results/{dataset}/{ann,cnn}_cv.csv`, `{ann,cnn}_cv_summary.json` |
| **Run time on Kaggle (estimate)** | ≈1.5 h typical, ≈3 h worst case |
| **Accelerator / Internet** | GPU T4 x2 / off |

Together with NB05a this replaces the former `05-experiments` notebook. Do not link `05-experiments` any more.

Step-by-step instructions for the whole pipeline: `Code/Newest/RUN_GUIDE.md`. Background: `Decision/012-split-notebooks-for-9h-sessions.md`.

In [ ]:
import os, random as _random, warnings, re, string, json, gc, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
SEED       = 42
MAX_VOCAB  = 50000
EMBED_DIM  = 300
DATASETS   = ["amazon", "yelp", "twitter"]

WORKING     = "/kaggle/working"
SHARED_DIR  = f"{WORKING}/shared_data"
RESULTS_DIR = f"{WORKING}/saved_results"
MODELS_DIR  = f"{WORKING}/saved_models"

def ensure_dirs(dataset=None):
    for base in [SHARED_DIR, RESULTS_DIR, MODELS_DIR]:
        os.makedirs(base, exist_ok=True)
        if dataset:
            os.makedirs(f"{base}/{dataset}", exist_ok=True)
        else:
            for ds in DATASETS:
                os.makedirs(f"{base}/{ds}", exist_ok=True)

def set_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    _random.seed(seed)
    np.random.seed(seed)

set_seeds()
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
    recall_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import wilcoxon

tf.keras.mixed_precision.set_global_policy("mixed_float16")
strategy = tf.distribute.MirroredStrategy()
N_GPUS = strategy.num_replicas_in_sync
print(f"MirroredStrategy: {N_GPUS} GPU(s)")

ANN_BATCH = 1024 * N_GPUS
CNN_BATCH = 128  * N_GPUS
print(f"ANN_BATCH={ANN_BATCH}  CNN_BATCH={CNN_BATCH}")

def set_seeds_tf(seed=SEED):
    set_seeds(seed)
    tf.random.set_seed(seed)

set_seeds_tf()
print(f"TF {tf.__version__}")

ensure_dirs()


In [ ]:
def evaluate(name, y_true, y_pred, y_prob=None):
    auc = round(roc_auc_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    ap  = round(average_precision_score(y_true, y_prob), 4) if y_prob is not None else float("nan")
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        "Model"        : name,
        "Accuracy"     : round(accuracy_score(y_true, y_pred), 4),
        "Precision"    : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall"       : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-score"     : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "Macro-F1"     : round(rep["macro avg"]["f1-score"], 4),
        "Weighted-F1"  : round(rep["weighted avg"]["f1-score"], 4),
        "MCC"          : round(matthews_corrcoef(y_true, y_pred), 4),
        "AUC-ROC"      : auc,
        "Avg-Precision": ap,
    }

def best_threshold(y_true, y_prob):
    prec, rec, thr = precision_recall_curve(y_true, y_prob)
    f1s = 2*prec*rec/(prec+rec+1e-9)
    idx = np.argmax(f1s[:-1])
    return float(thr[idx])

def make_es(patience=5, verbose=0):
    return EarlyStopping(monitor="val_loss", patience=patience,
                         restore_best_weights=True, verbose=verbose)

def make_tf_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

def make_seq_ds(X, y, batch, shuffle=True, seed=SEED):
    X = X.astype(np.int32)
    y = y.astype(np.float32)
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle: ds = ds.shuffle(min(len(y), 50000), seed=seed)
    return ds.batch(batch).cache().prefetch(tf.data.AUTOTUNE)

print("Helpers ready.")


In [ ]:
def build_ann(h1, h2, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(h2, activation="relu"),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def build_cnn(filters, lstm_units, lr, vocab_sz, emb_mat, max_len,
               seed=SEED, pretrained=True, use_bilstm=True, use_dropout=True):
    set_seeds_tf(seed)
    with strategy.scope():
        emb = (tf.keras.layers.Embedding(vocab_sz, EMBED_DIM,
                                          weights=[emb_mat], trainable=False)
               if pretrained else
               tf.keras.layers.Embedding(vocab_sz, EMBED_DIM, trainable=True))
        layers = [emb, tf.keras.layers.Conv1D(filters, 3, activation="relu"),
                  tf.keras.layers.MaxPooling1D(2)]
        if use_bilstm:
            layers.append(tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units)))
        else:
            layers.append(tf.keras.layers.GlobalMaxPooling1D())
        layers.append(tf.keras.layers.Dense(64, activation="relu"))
        if use_dropout:
            layers.append(tf.keras.layers.Dropout(0.5))
        layers.append(tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"))
        m = tf.keras.Sequential(layers)
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

def shallow_ann(h1, lr, seed=SEED):
    set_seeds_tf(seed)
    with strategy.scope():
        m = tf.keras.Sequential([
            tf.keras.layers.Dense(h1, activation="relu", input_shape=(EMBED_DIM,)),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(1, activation="sigmoid", dtype="float32"),
        ])
        m.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Model builders ready.")


In [ ]:
def load_dataset(dataset, nb01_shared):
    """Load all arrays for a given dataset."""
    d_dir = f"{nb01_shared}/{dataset}"
    cfg   = json.load(open(f"{d_dir}/config.json"))
    d = {
        "name"         : dataset,
        "cfg"          : cfg,
        "max_len"      : cfg["max_len"],
        "vocab_size"   : cfg["vocab_size"],
        "X_emb_train"  : np.load(f"{d_dir}/X_emb_train.npy"),
        "X_emb_val"    : np.load(f"{d_dir}/X_emb_val.npy"),
        "X_emb_test"   : np.load(f"{d_dir}/X_emb_test.npy"),
        "X_seq_train"  : np.load(f"{d_dir}/X_seq_train.npy"),
        "X_seq_val"    : np.load(f"{d_dir}/X_seq_val.npy"),
        "X_seq_test"   : np.load(f"{d_dir}/X_seq_test.npy"),
        "y_train"      : np.load(f"{d_dir}/y_train.npy"),
        "y_val"        : np.load(f"{d_dir}/y_val.npy"),
        "y_test"       : np.load(f"{d_dir}/y_test.npy"),
        "emb_matrix"   : np.load(f"{d_dir}/emb_matrix.npy"),
        "class_weights": {int(k):float(v) for k,v in
                          json.load(open(f"{d_dir}/class_weights.json")).items()},
    }
    with open(f"{d_dir}/test_texts_raw.txt", encoding="utf-8") as f:
        d["test_texts_raw"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_train.txt", encoding="utf-8") as f:
        d["X_txt_train"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_val.txt", encoding="utf-8") as f:
        d["X_txt_val"] = f.read().splitlines()
    with open(f"{d_dir}/X_txt_test.txt", encoding="utf-8") as f:
        d["X_txt_test"] = f.read().splitlines()
    print(f"  {dataset}: train={len(d['y_train']):,} val={len(d['y_val']):,} "
          f"test={len(d['y_test']):,} max_len={d['max_len']}")
    return d

def tv_combined(d):
    """Train+val combined for final training."""
    return {
        "X_emb": np.vstack([d["X_emb_train"], d["X_emb_val"]]),
        "X_seq": np.vstack([d["X_seq_train"], d["X_seq_val"]]),
        "y"    : np.concatenate([d["y_train"], d["y_val"]]),
        "X_txt": d["X_txt_train"] + d["X_txt_val"],
    }

print("Loader ready.")


## 1. Paths

In [ ]:
# ── UPDATE: replace username/slug with your actual Kaggle values ──────────────
NB01_DIR    = "/kaggle/input/notebooks/abhaysharma01702/01-preprocessing"
NB01_SHARED = f"{NB01_DIR}/shared_data"

print("Verifying NB01 shared_data...")
for ds in DATASETS:
    d = f"{NB01_SHARED}/{ds}"
    ok = os.path.isdir(d)
    n  = len(os.listdir(d)) if ok else 0
    cfg = json.load(open(f"{d}/config.json")) if ok else {}
    print(f"  {ds}: {'✓' if ok else '✗ MISSING'}  {n} files  "
          f"max_len={cfg.get('max_len','?')}  n_train={cfg.get('n_train','?')}")

NB02_DIR = "/kaggle/input/notebooks/abhaysharma01702/02-coa-ann"
NB03_DIR = "/kaggle/input/notebooks/abhaysharma01702/03-coa-cnn"
NB04_DIR = "/kaggle/input/notebooks/abhaysharma01702/04-final-training"
NB02_RESULTS = f"{NB02_DIR}/saved_results"
NB03_RESULTS = f"{NB03_DIR}/saved_results"
NB04_RESULTS = f"{NB04_DIR}/saved_results"

print("Verifying inputs...")
for ds in DATASETS:
    for rdir, name in [(NB02_RESULTS,"NB02"),(NB03_RESULTS,"NB03"),(NB04_RESULTS,"NB04")]:
        ok = os.path.isdir(f"{rdir}/{ds}")
        print(f"  {ds}/{name}: {'✓' if ok else '✗ MISSING'}")


In [ ]:
# ── Run control ───────────────────────────────────────────────────────────────
# This notebook is sized to finish well inside one 9-hour Kaggle session (see the header). As a safety net it saves
# every finished unit (one model/fold/seed) immediately and stops cleanly after TIME_LIMIT_HOURS. If that ever
# happens: save the version, add it as an input, list its slug in RESUME_SLUGS and run again — finished units are
# copied over and skipped.
import time, glob, shutil
RESUME_SLUGS     = []          # e.g. ["05b-cross-validation"]
TIME_LIMIT_HOURS = 8.0         # Kaggle sessions end at 9 h; the last hour is margin for loading and final cells
T_START = time.time()
def hours_used(): return (time.time() - T_START) / 3600

UNIT_DIR = f"{RESULTS_DIR}/units"
os.makedirs(UNIT_DIR, exist_ok=True)
for _slug in RESUME_SLUGS:
    _n = 0
    for _f in glob.glob(f"/kaggle/input/notebooks/abhaysharma01702/{_slug}/saved_results/units/*.json"):
        _dst = f"{UNIT_DIR}/{os.path.basename(_f)}"
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst); _n += 1
    print(f"Resumed {_n} finished units from {_slug}")

def unit_done(key): return os.path.exists(f"{UNIT_DIR}/{key}.json")
def unit_load(key):
    with open(f"{UNIT_DIR}/{key}.json") as fh:
        return json.load(fh)
def unit_save(key, obj):
    """Atomic write: a session killed mid-write cannot leave a truncated unit file behind."""
    tmp = f"{UNIT_DIR}/{key}.json.tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, f"{UNIT_DIR}/{key}.json")

class TimeLimit(Exception):
    pass
def check_time():
    if hours_used() > TIME_LIMIT_HOURS:
        raise TimeLimit()


## 2. Five-fold cross-validation (all three datasets)

In [ ]:
# ── 5-fold cross-validation of the fixed COA configurations ──────────────────
# 2026-10-12 fixes: (1) early stopping previously received np.zeros(...) as validation labels → now true labels;
# (2) early stopping now monitors a random 10% slice of each fold's TRAINING part (previously the scored fold).
# Folds, seeds (42+fold for ANN, 142+fold for CNN), epochs and patience are unchanged.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_all = {}
METRIC_KEYS = ["f1", "macro_f1", "mcc", "auc"]

def fold_scores(y_true, p):
    pred = (p > 0.5).astype(int)
    rep = classification_report(y_true, pred, output_dict=True, zero_division=0)
    return {"f1": float(f1_score(y_true, pred)), "macro_f1": rep["macro avg"]["f1-score"],
            "mcc": float(matthews_corrcoef(y_true, pred)), "auc": float(roc_auc_score(y_true, p))}

try:
    for ds in DATASETS:
        keys = [f"cv_{ds}_{a}_{k}" for a in ["ann", "cnn"] for k in range(1, 6)]
        if not all(unit_done(k) for k in keys):
            print(f"\n{'='*60}\n5-FOLD CV | {ds.upper()}\n{'='*60}")
            d = load_dataset(ds, NB01_SHARED)
            ml = d["max_len"]; vs = d["vocab_size"]; emb = d["emb_matrix"]
            ann_cfg = json.load(open(f"{NB02_RESULTS}/{ds}/coa_ann.json"))
            cnn_cfg = json.load(open(f"{NB03_RESULTS}/{ds}/coa_cnn.json"))
            h1=ann_cfg["best_h1"]; h2=ann_cfg["best_h2"]; ann_lr=ann_cfg["best_lr"]
            f_=cnn_cfg["best_filters"]; lstm_u=cnn_cfg["best_lstm_units"]; cnn_lr=cnn_cfg["best_lr"]
            X_emb_tv = np.vstack([d["X_emb_train"], d["X_emb_val"]])
            X_seq_tv = np.vstack([d["X_seq_train"], d["X_seq_val"]])
            y_tv = np.concatenate([d["y_train"], d["y_val"]])

            for fold, (tr_i, vl_i) in enumerate(skf.split(X_emb_tv, y_tv)):
                tr_perm = np.random.RandomState(SEED + fold).permutation(tr_i)
                n_es_f = int(len(tr_perm) * 0.1)
                fit_i, es_i = tr_perm[:-n_es_f], tr_perm[-n_es_f:]
                y_fit, y_es, y_vl_f = y_tv[fit_i], y_tv[es_i], y_tv[vl_i]
                cw_f = dict(zip([int(k) for k in np.unique(y_fit)],
                                [float(v) for v in compute_class_weight("balanced",
                                                    classes=np.unique(y_fit), y=y_fit)]))
                for arch in ["ann", "cnn"]:
                    key = f"cv_{ds}_{arch}_{fold+1}"
                    if unit_done(key): continue
                    check_time()
                    if arch == "ann":
                        m = build_ann(h1, h2, ann_lr, seed=SEED + fold)
                        m.fit(make_tf_ds(X_emb_tv[fit_i], y_fit, ANN_BATCH),
                              validation_data=make_tf_ds(X_emb_tv[es_i], y_es, ANN_BATCH, shuffle=False),
                              epochs=50, callbacks=[make_es(5)], class_weight=cw_f, verbose=0)
                        p = m.predict(make_tf_ds(X_emb_tv[vl_i], np.zeros(len(vl_i)), ANN_BATCH, shuffle=False),
                                      verbose=0).flatten()
                    else:
                        m = build_cnn(f_, lstm_u, cnn_lr, vs, emb, ml, seed=SEED + fold + 100)
                        m.fit(make_seq_ds(X_seq_tv[fit_i], y_fit, CNN_BATCH),
                              validation_data=make_seq_ds(X_seq_tv[es_i], y_es, CNN_BATCH, shuffle=False),
                              epochs=20, callbacks=[make_es(3)], class_weight=cw_f, verbose=0)
                        p = m.predict(make_seq_ds(X_seq_tv[vl_i], np.zeros(len(vl_i)), CNN_BATCH, shuffle=False),
                                      verbose=0).flatten()
                    s = fold_scores(y_vl_f, p)
                    unit_save(key, s)
                    print(f"  Fold {fold+1} {arch.upper()}: F1={s['f1']:.4f}  MCC={s['mcc']:.4f}  [{hours_used():.2f} h]")
                    del m; tf.keras.backend.clear_session(); gc.collect()
            del d, X_emb_tv, X_seq_tv, y_tv, emb; gc.collect()

        # Assemble per-dataset summaries in the same file format NB06 reads
        r_dir = f"{RESULTS_DIR}/{ds}"; os.makedirs(r_dir, exist_ok=True)
        cv_all[ds] = {}
        for key in ["ann", "cnn"]:
            folds = [unit_load(f"cv_{ds}_{key}_{k}") for k in range(1, 6) if unit_done(f"cv_{ds}_{key}_{k}")]
            cv = {m_: [f[m_] for f in folds] for m_ in METRIC_KEYS}
            cv_all[ds][key] = cv
            cv_df = pd.DataFrame(cv, index=[f"Fold {i+1}" for i in range(len(folds))])
            cv_df.loc["Mean"] = cv_df.mean().round(4); cv_df.loc["Std"] = cv_df.iloc[:len(folds)].std(ddof=0).round(4)
            cv_df.to_csv(f"{r_dir}/{key}_cv.csv")
            summary = {"model": key, "dataset": ds, "n_folds": len(folds),
                       "per_fold": {f"fold_{i+1}": {k: round(v[i], 4) for k, v in cv.items()} for i in range(len(folds))},
                       "summary": {k: {"mean": round(float(np.mean(v)), 4), "std": round(float(np.std(v)), 4)}
                                   for k, v in cv.items()}}
            json.dump(summary, open(f"{r_dir}/{key}_cv_summary.json", "w"), indent=2)
            print(f"  {ds} {key.upper()} CV ({len(folds)}/5 folds): F1={np.mean(cv['f1']):.4f}±{np.std(cv['f1']):.4f}")
except TimeLimit:
    print(f"\nTime limit ({TIME_LIMIT_HOURS} h) reached during CV. Finished folds are saved; "
          "link this version, set RESUME_SLUGS and run again.")

print("\nCV cell finished.")


## 3. Summary

In [ ]:
print("=== CV Summary (corrected protocol) ===\n")
for ds in DATASETS:
    print(ds.upper())
    for key in ["ann", "cnn"]:
        p = f"{RESULTS_DIR}/{ds}/{key}_cv_summary.json"
        if not os.path.exists(p):
            print(f"  COA-{key.upper():6s}: not finished"); continue
        s = json.load(open(p)); sm = s["summary"]
        print(f"  COA-{key.upper():6s} ({s['n_folds']}/5 folds): F1={sm['f1']['mean']:.4f}±{sm['f1']['std']:.4f}  "
              f"MCC={sm['mcc']['mean']:.4f}±{sm['mcc']['std']:.4f}")
    print()

print("Files saved:")
for ds in DATASETS:
    if os.path.isdir(f"{RESULTS_DIR}/{ds}"):
        for f_ in sorted(os.listdir(f"{RESULTS_DIR}/{ds}")):
            print(f"  {ds}/{f_}")
